# Chapter 6 Source Code Notebook

This notebook builds the source code for Chapter 6, **Coordinating Multiple Agents**.

The code builds a coordinated multi-agent system with Planner, Executor, Verifier, typed messages, shared state, and traceability for SupportOps AI.

Before running the notebook, install any required packages and set the required API keys in your environment where model powered examples are used.


## Step 1: Policy Rules File

### `policies/support_rules.yaml`

The multi-agent system begins with policy rules that every role must respect. These rules give the verifier deterministic standards to apply later in the workflow.

## Step 2: Typed Message Protocol

### `multi_agent/messages.py`

Next, the agents receive a shared message protocol. Each role communicates through typed objects so the handoff between Planner, Executor, and Verifier stays clear.

In [ ]:
# multi_agent/messages.py
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any, Optional

class MessageType(Enum):
    TASK_REQUEST     = "task_request"
    TASK_PLAN        = "task_plan"
    EXECUTE_REQUEST  = "execute_request"
    EXECUTION_RESULT = "execution_result"
    VERIFY_REQUEST   = "verify_request"
    VERIFY_VERDICT   = "verify_verdict"

class VerdictType(Enum):
    APPROVED             = "approved"
    REJECTED             = "rejected"
    APPROVED_CONDITIONAL = "approved_conditional"

@dataclass
class RequiredChange:
    rule_id:     str
    description: str
    action:      str
    severity:    str   # low | medium | high | critical

@dataclass
class AgentMessage:
    """Base class - every message carries correlation and timestamp."""
    correlation_id: str
    msg_type:       MessageType
    sender_role:    str
    sent_at:        str = field(
        default_factory=lambda: datetime.utcnow().isoformat()
    )

@dataclass
class TaskRequest(AgentMessage):
    ticket_id:       str = ""
    raw_input:       str = ""
    session_summary: dict = field(default_factory=dict)

@dataclass
class TaskPlan(AgentMessage):
    plan_id:    str = ""
    hypothesis: str = ""
    steps:      list = field(default_factory=list)
    confidence: str = "medium"
    reasoning:  str = ""

@dataclass
class ExecutionResult(AgentMessage):
    plan_id:       str = ""
    status:        str = ""
    draft_response: Optional[str] = None
    actions_taken: list = field(default_factory=list)
    cost_usd:      float = 0.0
    error:         Optional[str] = None

@dataclass
class VerificationVerdict(AgentMessage):
    verdict:              VerdictType = VerdictType.APPROVED
    issues:               list[RequiredChange] = field(default_factory=list)
    approved_response:    Optional[str] = None
    policy_version_used:  str = ""
    notes:                str = ""


## Step 3: Versioned Shared State

### `multi_agent/versioned_state.py`

With messages defined, the shared state adds version control around the information agents depend on. This prevents stale writes and makes state changes easier to reason about.

In [ ]:
# multi_agent/versioned_state.py
import threading
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Optional

@dataclass
class StateVersion:
    version:    int
    written_by: str       # agent role that triggered the write
    written_at: str
    field_name: str
    old_value:  Any
    new_value:  Any

class VersionedSessionState:
    """
    Thread-safe versioned state store.
    Only the Orchestrator writes; agents receive read-only snapshots.
    """
    def __init__(self, ticket_id: str, raw_input: str):
        self._lock    = threading.Lock()
        self._version = 0
        self._history: list[StateVersion] = []
        self._state: dict[str, Any] = {
            "ticket_id":      ticket_id,
            "raw_input":      raw_input,
            "status":         "open",
            "classification": None,
            "routing":        None,
            "draft_response": None,
            "escalated":      False,
            "escalation_reason": None,
            "final_response": None,
        }

    @property
    def version(self) -> int:
        return self._version

    def snapshot(self) -> tuple[dict, int]:
        """Return (current_state_copy, current_version)."""
        with self._lock:
            import copy
            return copy.deepcopy(self._state), self._version

    def write(self, field: str, value: Any,
               writer_role: str,
               expected_version: int) -> tuple[bool, str]:
        """
        Optimistic write. Returns (success, reason).
        Fails if current version != expected_version (stale write).
        """
        with self._lock:
            if self._version != expected_version:
                return False, (
                    f"Stale write: expected v{expected_version}, "
                    f"current v{self._version}"
                )
            old = self._state.get(field)
            self._state[field] = value
            self._version += 1
            self._history.append(StateVersion(
                version    = self._version,
                written_by = writer_role,
                written_at = datetime.utcnow().isoformat(),
                field_name = field,
                old_value  = old,
                new_value  = value
            ))
            return True, "ok"

    def audit_trail(self) -> list[StateVersion]:
        with self._lock:
            return list(self._history)


## Step 4: Multi-Agent Trace Logger

### `multi_agent/trace.py`

The trace logger then links events across agents with correlation identifiers. It helps you follow a task as it moves through planning, execution, verification, and orchestration.

In [ ]:
# multi_agent/trace.py
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any

@dataclass
class TraceEntry:
    correlation_id: str
    agent_role:     str      # planner | executor | verifier | orchestrator
    event_type:     str      # message_received | decision_made |
                             # tool_called | message_sent | state_written
    timestamp:      str = field(
        default_factory=lambda: datetime.utcnow().isoformat()
    )
    payload:        dict = field(default_factory=dict)

class MultiAgentTraceLogger:
    def __init__(self, correlation_id: str):
        self.correlation_id = correlation_id
        self._entries: list[TraceEntry] = []

    def log(self, agent_role: str, event_type: str,
             payload: dict) -> TraceEntry:
        entry = TraceEntry(
            correlation_id = self.correlation_id,
            agent_role     = agent_role,
            event_type     = event_type,
            payload        = payload
        )
        self._entries.append(entry)
        return entry

    def timeline(self) -> str:
        """Human-readable multi-agent timeline."""
        lines = [
            f"Multi-Agent Timeline - Correlation: {self.correlation_id}",
            "=" * 60, ""
        ]
        for e in self._entries:
            ts = e.timestamp[11:19]  # HH:MM:SS
            role_pad = e.agent_role.upper().ljust(12)
            event_pad = e.event_type.ljust(18)
            summary = _payload_summary(e.payload)
            lines.append(f"  {ts}  [{role_pad}]  {event_pad}  {summary}")
        lines.append("")
        return "\n".join(lines)

    def entries_for_role(self, role: str) -> list[TraceEntry]:
        return [e for e in self._entries if e.agent_role == role]

def _payload_summary(payload: dict) -> str:
    """One-line summary of a payload dict for timeline display."""
    key_fields = ["plan_id", "verdict", "status", "action",
                   "rule_id", "error", "msg_type"]
    parts = []
    for k in key_fields:
        if k in payload:
            parts.append(f"{k}={payload[k]}")
    return ", ".join(parts) if parts else str(payload)[:60]


## Step 5: The Three Agents

### `multi_agent/agents.py`

Now the three specialist agents are implemented with separate responsibilities. The Planner decides what should happen, the Executor performs the work, and the Verifier checks the result.

In [ ]:
# multi_agent/agents.py
import json, yaml
import anthropic
from ..tracker import tracked_call, SystemTrace
from ..config import SystemConfig, DEFAULT_CONFIG
from .messages import (
    TaskRequest, TaskPlan, ExecutionResult,
    VerificationVerdict, VerdictType, RequiredChange, MessageType
)
from .trace import MultiAgentTraceLogger

# ── Planner Agent ─────────────────────────────────────────────────────
PLANNER_PROMPT = """
You are the Planner agent. Your only job: produce a structured task plan.
Do NOT execute actions. Do NOT verify policy. Plan only.

Ticket: {ticket}
Session: {session_summary}
Available tools: {tools}

Return JSON:
{
  "plan_id": "P-<short id>",
  "hypothesis": "<one sentence>",
  "steps": [{"action": "<tool>", "inputs": {{}},
              "description": "<why>", "reversible": true}],
  "confidence": "high|medium|low",
  "reasoning": "<two sentences max>"
}
Return ONLY valid JSON."""

class PlannerAgent:
    def __init__(self, client: anthropic.Anthropic,
                 tool_names: list[str],
                 config: SystemConfig = DEFAULT_CONFIG):
        self.client     = client
        self.tool_names = tool_names
        self.config     = config

    def run(self, request: TaskRequest,
             sys_trace: SystemTrace,
             ma_trace: MultiAgentTraceLogger) -> TaskPlan:
        ma_trace.log("planner", "message_received",
            {"msg_type": request.msg_type.value,
             "ticket_id": request.ticket_id})
        prompt = PLANNER_PROMPT.format(
            ticket=request.raw_input[:500],
            session_summary=json.dumps(request.session_summary),
            tools=", ".join(self.tool_names)
        )
        response = tracked_call(
            self.client, "planner", sys_trace,
            model=self.config.smart_model, max_tokens=512,
            messages=[{"role": "user", "content": prompt}]
        )
        data = json.loads(response.content[0].text.strip())
        plan = TaskPlan(
            correlation_id = request.correlation_id,
            msg_type       = MessageType.TASK_PLAN,
            sender_role    = "planner",
            plan_id        = data["plan_id"],
            hypothesis     = data["hypothesis"],
            steps          = data["steps"],
            confidence     = data.get("confidence", "medium"),
            reasoning      = data.get("reasoning", "")
        )
        ma_trace.log("planner", "message_sent",
            {"msg_type": plan.msg_type.value,
             "plan_id": plan.plan_id,
             "steps": len(plan.steps),
             "confidence": plan.confidence})
        return plan

# ── Executor Agent ────────────────────────────────────────────────────
class ExecutorAgent:
    def __init__(self, registry,
                 config: SystemConfig = DEFAULT_CONFIG):
        self.registry = registry
        self.config   = config
        self._executed: set = set()  # idempotency tracking

    def run(self, plan: TaskPlan,
             session_snapshot: dict,
             sys_trace: SystemTrace,
             ma_trace: MultiAgentTraceLogger) -> ExecutionResult:
        ma_trace.log("executor", "message_received",
            {"msg_type": MessageType.TASK_PLAN.value,
             "plan_id": plan.plan_id,
             "steps": len(plan.steps)})
        actions = []
        draft   = None
        total_cost = 0.0

        for step in plan.steps:
            action = step.get("action", "")
            inputs = step.get("inputs", {})
            # Idempotency check
            sig = f"{plan.plan_id}:{action}:{sorted(inputs.items())}"
            if sig in self._executed:
                ma_trace.log("executor", "decision_made",
                    {"action": action, "decision": "skip_duplicate"})
                continue
            self._executed.add(sig)
            result = self.registry.execute(action, inputs)
            ma_trace.log("executor", "tool_called",
                {"action": action, "success": result.success,
                 "error": result.error})
            actions.append({
                "action": action, "success": result.success,
                "error": result.error
            })
            if not result.success:
                return ExecutionResult(
                    correlation_id = plan.correlation_id,
                    msg_type       = MessageType.EXECUTION_RESULT,
                    sender_role    = "executor",
                    plan_id        = plan.plan_id,
                    status         = "failed",
                    actions_taken  = actions,
                    error          = f"Step '{action}' failed: {result.error}"
                )
            if action == "draft_response":
                draft = str(result.data)
        result_msg = ExecutionResult(
            correlation_id = plan.correlation_id,
            msg_type       = MessageType.EXECUTION_RESULT,
            sender_role    = "executor",
            plan_id        = plan.plan_id,
            status         = "success",
            draft_response = draft,
            actions_taken  = actions,
            cost_usd       = total_cost
        )
        ma_trace.log("executor", "message_sent",
            {"msg_type": result_msg.msg_type.value,
             "status": result_msg.status,
             "draft_len": len(draft) if draft else 0})
        return result_msg

# ── Verifier Agent ────────────────────────────────────────────────────
class VerifierAgent:
    def __init__(self, policy_path: str):
        with open(policy_path) as f:
            policy_doc = yaml.safe_load(f)
        self.rules         = policy_doc["rules"]
        self.policy_version = policy_doc["version"]

    def run(self, result: ExecutionResult,
             session_snapshot: dict,
             ma_trace: MultiAgentTraceLogger) -> VerificationVerdict:
        ma_trace.log("verifier", "message_received",
            {"msg_type": MessageType.EXECUTION_RESULT.value,
             "plan_id": result.plan_id,
             "has_draft": result.draft_response is not None})
        issues = []
        draft  = result.draft_response or ""
        for rule in self.rules:
            violation = self._check_rule(rule, draft, session_snapshot)
            if violation:
                issues.append(RequiredChange(
                    rule_id     = rule["id"],
                    description = rule["description"],
                    action      = rule["required_action"],
                    severity    = rule["severity"]
                ))
                ma_trace.log("verifier", "decision_made",
                    {"rule_id": rule["id"],
                     "verdict": "violation",
                     "severity": rule["severity"]})
        if not issues:
            verdict = VerdictType.APPROVED
        elif all(i.severity in ("low",) for i in issues):
            verdict = VerdictType.APPROVED_CONDITIONAL
        else:
            verdict = VerdictType.REJECTED
        msg = VerificationVerdict(
            correlation_id      = result.correlation_id,
            msg_type            = MessageType.VERIFY_VERDICT,
            sender_role         = "verifier",
            verdict             = verdict,
            issues              = issues,
            approved_response   = draft if verdict != VerdictType.REJECTED
                                 else None,
            policy_version_used = self.policy_version
        )
        ma_trace.log("verifier", "message_sent",
            {"msg_type": msg.msg_type.value,
             "verdict": verdict.value,
             "issues_count": len(issues)})
        return msg

    def _check_rule(self, rule: dict, draft: str,
                     session: dict) -> bool:
        """Returns True if rule is violated."""
        rid = rule["id"]
        if rid == "R-01":
            import re
            return bool(re.search(r'\$[0-9]+', draft))
        if rid == "R-02":
            legal_words = ["lawsuit", "attorney", "legal action",
                           "sue ", "lawyer"]
            has_legal = any(w in draft.lower() for w in legal_words)
            escalated = session.get("escalated", False)
            return has_legal and not escalated
        if rid == "R-03":
            import re
            return bool(re.search(
                r'\b(by|within|in)\s+(\d+)\s+(hour|day|minute)',
                draft.lower()
            ))
        if rid == "R-04":
            escalated = session.get("escalated", False)
            resolution_words = ["i have resolved", "issue is fixed",
                                "refund has been issued"]
            return escalated and any(
                w in draft.lower() for w in resolution_words
            )
        if rid == "R-05":
            ticket_id = session.get("ticket_id", "")
            return ticket_id not in draft
        return False


## Step 6: The Orchestrator

### `multi_agent/orchestrator.py`

The orchestrator connects the agents without turning coordination into another model-driven decision. It routes messages, owns shared state updates, and keeps the overall sequence deterministic.

In [ ]:
# multi_agent/orchestrator.py
import os
import anthropic
from .agents import PlannerAgent, ExecutorAgent, VerifierAgent
from .messages import (
    TaskRequest, MessageType, VerdictType
)
from .versioned_state import VersionedSessionState
from .trace import MultiAgentTraceLogger
from ..tracker import SystemTrace
from ..config import DEFAULT_CONFIG

MAX_VERIFY_CYCLES = 2

class MultiAgentOrchestrator:
    def __init__(self, client: anthropic.Anthropic,
                 registry, policy_path: str,
                 config=DEFAULT_CONFIG):
        tool_names   = [s.name for s in registry.all_specs()]
        self.planner  = PlannerAgent(client, tool_names, config)
        self.executor = ExecutorAgent(registry, config)
        self.verifier = VerifierAgent(policy_path)
        self.config   = config

    def run(self, ticket_id: str,
             raw_input: str) -> tuple[VersionedSessionState,
                                       MultiAgentTraceLogger]:
        state   = VersionedSessionState(ticket_id, raw_input)
        ma_trace = MultiAgentTraceLogger(ticket_id)
        sys_trace = SystemTrace(ticket_id)

        ma_trace.log("orchestrator", "message_received",
            {"ticket_id": ticket_id,
             "input_len": len(raw_input)})

        # ── Phase 1: Plan ────────────────────────────────────────────
        snap, ver = state.snapshot()
        request = TaskRequest(
            correlation_id = ticket_id,
            msg_type       = MessageType.TASK_REQUEST,
            sender_role    = "orchestrator",
            ticket_id      = ticket_id,
            raw_input      = raw_input,
            session_summary = snap
        )
        plan = self.planner.run(request, sys_trace, ma_trace)
        state.write("plan", plan, "orchestrator", ver)
        _, ver = state.snapshot()
        ma_trace.log("orchestrator", "state_written",
            {"field": "plan", "version": ver,
             "plan_id": plan.plan_id})

        # ── Phase 2: Execute + Verify cycle ─────────────────────────
        for cycle in range(MAX_VERIFY_CYCLES):
            snap, ver = state.snapshot()
            result = self.executor.run(plan, snap, sys_trace, ma_trace)
            if result.status == "failed":
                ma_trace.log("orchestrator", "decision_made",
                    {"decision": "escalate",
                     "reason": f"execution_failed: {result.error}"})
                state.write("status", "escalated", "orchestrator", ver)
                state.write("escalation_reason",
                    f"execution_failed: {result.error}",
                    "orchestrator", ver + 1)
                return state, ma_trace

            snap, ver = state.snapshot()
            verdict = self.verifier.run(result, snap, ma_trace)

            if verdict.verdict == VerdictType.APPROVED:
                ok, _ = state.write(
                    "final_response", verdict.approved_response,
                    "orchestrator", ver
                )
                state.write("status", "resolved",
                    "orchestrator", ver + (1 if ok else 0))
                ma_trace.log("orchestrator", "decision_made",
                    {"decision": "resolved",
                     "cycle": cycle + 1})
                return state, ma_trace

            if verdict.verdict == VerdictType.APPROVED_CONDITIONAL:
                ok, _ = state.write(
                    "final_response", verdict.approved_response,
                    "orchestrator", ver
                )
                state.write("status", "resolved_with_notes",
                    "orchestrator", ver + (1 if ok else 0))
                ma_trace.log("orchestrator", "decision_made",
                    {"decision": "resolved_conditional",
                     "low_issues": len(verdict.issues)})
                return state, ma_trace

            # REJECTED: feed issues back if cycles remain
            if cycle < MAX_VERIFY_CYCLES - 1:
                issues_summary = [
                    f"{i.rule_id}: {i.action}" for i in verdict.issues
                ]
                plan.reasoning += (
                    f" [Retry {cycle+1}: verifier rejected - ",
                    f"{issues_summary}]"
                )
                ma_trace.log("orchestrator", "decision_made",
                    {"decision": "retry_execution",
                     "cycle": cycle + 1,
                     "issues": issues_summary})

        # Cycle limit reached
        snap, ver = state.snapshot()
        state.write("status", "escalated", "orchestrator", ver)
        state.write("escalation_reason",
            "verify_cycle_limit_reached", "orchestrator", ver + 1)
        ma_trace.log("orchestrator", "decision_made",
            {"decision": "escalate",
             "reason": "verify_cycle_limit_reached"})
        return state, ma_trace


## Step 7: Running the Multi-Agent System

### `multi_agent/run.py`

The final cell runs the complete multi-agent workflow. It shows how the policy rules, messages, state, trace, agents, and orchestrator work together.

In [ ]:
# multi_agent/run.py
import os
import anthropic
from supportops_ai.multi_agent.orchestrator import MultiAgentOrchestrator
from supportops_ai.agent.run_agent import registry

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

orchestrator = MultiAgentOrchestrator(
    client      = client,
    registry    = registry,
    policy_path = "policies/support_rules.yaml"
)

if __name__ == "__main__":
    ticket = (
        "I've been charged twice this month and I spoke to someone "
        "yesterday who said I'd get a $47.50 refund by Tuesday. "
        "I haven't received it. If this isn't resolved today "
        "I'm contacting my attorney."
    )
    state, trace = orchestrator.run("TKT-008", ticket)

    print(trace.timeline())
    print("\n=== FINAL STATE ===")
    snap, ver = state.snapshot()
    print(f"Status:   {snap['status']}")
    print(f"Version:  {ver}")
    if snap.get('final_response'):
        print(f"\nFinal Response:\n{snap['final_response']}")
    if snap.get('escalation_reason'):
        print(f"\nEscalation reason: {snap['escalation_reason']}")

    print("\n=== STATE AUDIT TRAIL ===")
    for sv in state.audit_trail():
        print(f"  v{sv.version} [{sv.written_by}] ",
              f"{sv.field_name} = {str(sv.new_value)[:60]}")
